# SAS to Python: Finding Missing Rows

This notebook demonstrates how to debug missing rows after a data migration using set differences and `dataprof`.

In [ ]:
# Run this cell first to install dependencies in Colab
!pip install pandas dataprof


## Generate Dummy Data

Run the cell below to create `mother_db.csv` and `broken_subset_db.csv` for our exercise.

In [ ]:
import pandas as pd
import numpy as np
import random

# Create dummy 'mother_db'
np.random.seed(42)
num_rows = 1000
mother_data = {
    'contract_id': [f'C{i:05d}' for i in range(1, num_rows + 1)],
    'condition': np.random.choice([True, False], size=num_rows, p=[0.7, 0.3]),
    'value': np.random.normal(100, 15, size=num_rows),
    'YOUR_SAMPLE_COLUMN': [random.choice([np.nan, 'A', 'B', 'C']) for _ in range(num_rows)]
}
mother_db = pd.DataFrame(mother_data)
mother_db.to_csv("mother_db.csv", index=False)

# Create dummy 'broken_subset_db' (missing some rows, some NaNs)
subset_data = mother_db[mother_db['condition']].copy()
subset_data = subset_data.drop(subset_data.sample(40, random_state=42).index)
subset_data.loc[subset_data.sample(10).index, 'YOUR_SAMPLE_COLUMN'] = np.nan
subset_data.to_csv("broken_subset_db.csv", index=False)

print("Created dummy data: 'mother_db.csv' and 'broken_subset_db.csv'")


## Debugging the Missing Rows

Find which rows disappeared using set differences, and then profile the datsets using `dataprof`.

In [ ]:
import pandas as pd
import dataprof as dp

# Load your dataframes
mother_db = pd.read_csv("mother_db.csv")  # Load your main dataframe
broken_subset_db = pd.read_csv("broken_subset_db.csv")  # Load the subset that's missing rows

# 1. Get the IDs that SHOULD be in the perimeter (from the mother DB)
expected_ids = set(mother_db[mother_db['condition']]['contract_id'])

# 2. Get the IDs that actually MADE IT into his new sub-DB
actual_ids = set(broken_subset_db['contract_id'])

# 3. Find the missing ones!
missing_ids = expected_ids - actual_ids
print(f"Missing {len(missing_ids)} IDs:\n")

# 4. Print the exact rows that vanished to see what they have in common
missing_rows = mother_db[mother_db['contract_id'].isin(missing_ids)]
print("Vanished rows:")
display(missing_rows)

print("\n" + "="*40 + "\n")

# 5. Profile the Mother DB
report_mother = dp.profile(mother_db) # Pass your main dataframe here
print("--- MOTHER DB PROFILE ---")
print(f"Total rows: {report_mother.rows}")
print(f"Nulls in 'YOUR_SAMPLE_COLUMN': {report_mother.column_profiles['YOUR_SAMPLE_COLUMN'].null_count}")

# 6. Profile one of the broken subsets (e.g. where 40 rows are missing)
report_subset = dp.profile(broken_subset_db)
print("\n--- SUBSET DB PROFILE ---")
print(f"Total rows: {report_subset.rows}")

# 7. Export the reports to JSON so you can compare them easily
report_mother.save("profile_mother.json")
report_subset.save("profile_subset.json")
print("\nSaved profiles to JSON.")
